# Notebook 01: Procesos, Hilos y el GIL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/16_computo/code/01_procesos_hilos.ipynb)

**Módulo 16 — Clase 1**

Este notebook acompaña los archivos `01_procesos_y_hilos.md` y `02_secuencial.md`.

Secciones **[EN CLASE]** se trabajan durante la sesión.  
Secciones **[TAREA]** se completan después.

---

In [ ]:
import os
import sys
import time
import threading
import multiprocessing

print(f'Python {sys.version}')
print(f'CPU cores: {os.cpu_count()}')

## [EN CLASE] Sección 1: Inspeccionando un proceso

Cada programa Python que ejecutas ES un proceso con su propio PID y espacio de memoria.

In [ ]:
print(f'Mi PID (Process ID): {os.getpid()}')
print(f'PID de mi proceso padre: {os.getppid()}')

# Intentar instalar psutil si no está disponible
try:
    import psutil
    proc = psutil.Process(os.getpid())
    mem = proc.memory_info()
    print(f'Memoria RSS: {mem.rss / 1024**2:.1f} MB')
    print(f'Memoria VMS: {mem.vms / 1024**2:.1f} MB')
except ImportError:
    print('psutil no instalado — pip install psutil para más info')

## [EN CLASE] Sección 2: Inspeccionando hilos

El hilo principal es θ_main. Podemos crear hilos adicionales — todos comparten el heap del proceso.

In [ ]:
print(f'Hilo actual: {threading.current_thread().name}')
print(f'ID del hilo: {threading.current_thread().ident}')
print(f'Hilos activos: {threading.active_count()}')

# Crear un hilo que reporte su identidad
def reportar_identidad(nombre):
    print(f'  [{nombre}] PID={os.getpid()}, Hilo={threading.current_thread().name}, ID={threading.current_thread().ident}')

print('\nLanzando 3 hilos:')
hilos = []
for i in range(3):
    h = threading.Thread(target=reportar_identidad, args=(f'hilo-{i}',))
    hilos.append(h)
    h.start()

for h in hilos:
    h.join()

print('\nObserva: todos los hilos tienen el MISMO PID — son del mismo proceso.')

## [EN CLASE] Sección 3: Demostración del GIL — CPU-bound con threading

**Hipótesis:** con 2 hilos y 2 cores, la tarea CPU-bound debería tardar la mitad.  
**Resultado real:** el GIL serializa los hilos → casi no hay speedup (puede ser más lento).

In [ ]:
def tarea_cpu_bound(n, nombre=''):
    """Tarea CPU-bound pura: suma de 0 a n. wait(τ) = ∅"""
    resultado = sum(range(n))
    return resultado

N = 30_000_000  # ajusta si es muy lento o muy rápido en tu máquina

# --- Secuencial ---
t0 = time.perf_counter()
tarea_cpu_bound(N)
tarea_cpu_bound(N)
t_secuencial = time.perf_counter() - t0

# --- Threading (2 hilos, misma carga total) ---
t0 = time.perf_counter()
h1 = threading.Thread(target=tarea_cpu_bound, args=(N,))
h2 = threading.Thread(target=tarea_cpu_bound, args=(N,))
h1.start(); h2.start()
h1.join(); h2.join()
t_threading = time.perf_counter() - t0

print(f'Secuencial:  {t_secuencial:.2f}s')
print(f'Threading:   {t_threading:.2f}s')
print(f'Speedup:     {t_secuencial/t_threading:.2f}x  (esperado ~2x, real ~1x por el GIL)')
print()
print('Conclusión: para CPU-bound, threading en Python NO ayuda.')
print('El GIL garantiza que solo un hilo ejecuta bytecode a la vez.')

---

## [TAREA] Sección 4: GIL liberado — I/O-bound con threading

Durante `wait(τᵢ)` (I/O), el GIL se libera. Aquí sí hay speedup con threading.

In [ ]:
def tarea_io_bound(duracion, nombre=''):
    """Simula I/O-bound: espera bloqueante. wait(τ) ≠ ∅"""
    time.sleep(duracion)  # durante sleep, el GIL se libera

DURACION = 1.0  # cada tarea "espera" 1s de I/O

# --- Secuencial ---
t0 = time.perf_counter()
tarea_io_bound(DURACION)
tarea_io_bound(DURACION)
t_secuencial = time.perf_counter() - t0

# --- Threading ---
t0 = time.perf_counter()
h1 = threading.Thread(target=tarea_io_bound, args=(DURACION,))
h2 = threading.Thread(target=tarea_io_bound, args=(DURACION,))
h1.start(); h2.start()
h1.join(); h2.join()
t_threading = time.perf_counter() - t0

print(f'Secuencial:  {t_secuencial:.2f}s  (suma: 2 × {DURACION}s)')
print(f'Threading:   {t_threading:.2f}s  (paralelo en I/O, GIL liberado)')
print(f'Speedup:     {t_secuencial/t_threading:.2f}x  (esperado ~2x)')
print()
print('Conclusión: para I/O-bound, threading SÍ ayuda porque el GIL se libera durante sleep/I/O.')

## [TAREA] Sección 5: Aislamiento de memoria — proceso vs hilo

Los hilos comparten memoria (un cambio es visible para todos).  
Los procesos NO comparten memoria (cada uno tiene su propia copia).

In [ ]:
# --- Hilos comparten memoria ---
contador_compartido = [0]  # lista mutable accesible por todos los hilos

def incrementar_hilo(n):
    for _ in range(n):
        contador_compartido[0] += 1  # AMBOS hilos ven y modifican esto

hilos = [threading.Thread(target=incrementar_hilo, args=(1000,)) for _ in range(2)]
for h in hilos: h.start()
for h in hilos: h.join()

print(f'Contador tras 2 hilos × 1000 incrementos: {contador_compartido[0]}')
print('(Nota: puede ser < 2000 debido a race condition — lo veremos en Clase 2)')

print()

# --- Procesos NO comparten memoria ---
def incrementar_proceso(lista_compartida, n):
    # En un proceso hijo, lista_compartida es una COPIA — no afecta al padre
    for _ in range(n):
        lista_compartida[0] += 1
    print(f'  Proceso hijo ({os.getpid()}): su copia = {lista_compartida[0]}')

if __name__ == '__main__':  # necesario para multiprocessing en scripts
    dato = [0]
    p = multiprocessing.Process(target=incrementar_proceso, args=(dato, 1000))
    p.start()
    p.join()
    print(f'Proceso padre ({os.getpid()}): su copia = {dato[0]}')
    print('→ El padre no ve el cambio del hijo: Mem(padre) ∩ Mem(hijo) = ∅')

---

## Resumen

| Concepto | Clave |
|---|---|
| Proceso | Tiene su propia memoria aislada |
| Hilo | Comparte memoria del proceso |
| GIL | Solo 1 hilo ejecuta bytecode Python a la vez |
| CPU-bound + threading | Sin speedup (GIL nunca se libera) |
| I/O-bound + threading | Con speedup (GIL se libera durante I/O) |